# Персонализация Stable Diffusion XL через LoRA

Задача: научить SDXL генерировать конкретного человека по триггер-слову,
обучаясь на небольшом наборе фотографий.

Обучаются только LoRA-адаптеры в attention-слоях UNet — базовые веса (около
2.6 млрд параметров) заморожены, обновляется меньше процента. Текстовые
энкодеры не затрагиваются.

## Про изображения в этом ноутбуке

Модель обучена на фотографиях реального человека. Ни исходные снимки, ни
сгенерированные портреты, ни файл весов в репозиторий не выкладываются — это
чужие персональные данные, и портрет, синтезированный моделью, остаётся
изображением конкретного человека.

Поэтому оценка здесь количественная: вместо галереи картинок считается
косинусная близость эмбеддингов лиц между генерациями и обучающими фото.
Генерации создаются и измеряются, но не отображаются.

Чтобы воспроизвести на своих данных, достаточно положить 10-20 фотографий
одного человека — см. README.


## Окружение


In [ ]:
!nvidia-smi

Mon May 18 21:04:43 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.105.08             Driver Version: 580.105.08     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |       

... [лог сокращён] ...

I   CI              PID   Type   Process name                        GPU Memory |
|        ID   ID                                                               Usage      |
|=========================================================================================|
|  No running processes found                                                             |
+-----------------------------------------------------------------------------------------+


In [ ]:
!pip install -q -U git+https://github.com/huggingface/diffusers
!pip install -q -U git+https://github.com/huggingface/peft
!pip install -q -U accelerate transformers xformers datasets bitsandbytes

In [ ]:
import os
import json
import shutil

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

SEED = 42
TRIGGER = "mylza"          # редкий токен: не разбирается токенизатором на осмысленные части
BASE_MODEL = "stabilityai/stable-diffusion-xl-base-1.0"
TRAIN_DIR = "/kaggle/working/training_data"
OUTPUT_DIR = "/kaggle/working/lora_output"

torch.manual_seed(SEED)
np.random.seed(SEED)

## Подготовка датасета

Каждой фотографии нужна подпись. Использую одну и ту же короткую фразу
`a photo of mylza woman`. Описывать позу, фон и одежду не нужно — наоборот:
чем короче подпись, тем сильнее модель связывает триггер именно с внешностью,
а не с обстановкой кадра.

Триггер подобран так, чтобы токенизатор не разложил его на осмысленные части и
адаптер не перетянул на себя уже выученное понятие.


In [ ]:
# путь к своим фотографиям: 10-20 снимков одного человека
SOURCE_DIR = "/kaggle/input/datasets/<user>/<dataset>/photos"

os.makedirs(TRAIN_DIR, exist_ok=True)
for filename in os.listdir(SOURCE_DIR):
    if filename.lower().endswith((".png", ".jpg", ".jpeg")):
        shutil.copy(os.path.join(SOURCE_DIR, filename), os.path.join(TRAIN_DIR, filename))

with open(os.path.join(TRAIN_DIR, "metadata.jsonl"), "w") as f:
    for filename in sorted(os.listdir(TRAIN_DIR)):
        if filename.lower().endswith((".png", ".jpg", ".jpeg")):
            f.write(json.dumps({"file_name": filename,
                                "text": f"a photo of {TRIGGER} woman"}) + "\n")

n_images = len([f for f in os.listdir(TRAIN_DIR)
                if f.lower().endswith((".png", ".jpg", ".jpeg"))])
print(f"Изображений в обучающей выборке: {n_images}")

Изображений в обучающей выборке: 12

## Обучение

Скрипт `train_text_to_image_lora_sdxl.py` берётся из `diffusers` без изменений —
переписывать эталонную реализацию смысла нет.

Разбор параметров:

- `resolution=1024` — родное разрешение SDXL, при меньшем теряются детали лица.
- `train_batch_size=1` плюс `gradient_accumulation_steps=4` — эффективный батч 4,
  больше в 16 ГБ не помещается даже с оптимизациями.
- `gradient_checkpointing` — пересчёт активаций вместо их хранения. Платим
  примерно 30% времени за саму возможность запуститься.
- `use_8bit_adam` — состояния оптимизатора в 8 бит вместо 32; для Adam это два
  тензора на параметр, экономия ощутимая.
- `mixed_precision=fp16` и `xformers` — по той же причине.

2000 шагов при эффективном батче 4 и 12 изображениях — это порядка 660 проходов
по набору. Риск переобучения высокий, поэтому чекпоинты пишутся каждые 250 шагов.


In [ ]:
!wget -q https://raw.githubusercontent.com/huggingface/diffusers/main/examples/text_to_image/train_text_to_image_lora_sdxl.py

In [ ]:
!python train_text_to_image_lora_sdxl.py \
  --pretrained_model_name_or_path="stabilityai/stable-diffusion-xl-base-1.0" \
  --pretrained_vae_model_name_or_path="madebyollin/sdxl-vae-fp16-fix" \
  --train_data_dir="/kaggle/working/training_data" \
  --resolution=1024 \
  --train_batch_size=1 \
  --gradient_accumulation_steps=4 \
  --gradient_checkpointing \
  --use_8bit_adam \
  --learning_rate=5e-5 \
  --lr_scheduler="constant" \
  --lr_warmup_steps=0 \
  --mixed_precision="fp16" \
  --max_train_steps=2000 \
  --checkpointing_steps=250 \
  --output_dir="/kaggle/working/lora_output" \
  --seed=42 \
  --enable_xformers_memory_efficient_attention

Steps: 100%|██| 2000/2000 [5:07:37<00:00,  9.23s/it, lr=5e-5, step_loss=0.00114]

Обучение заняло 5 часов 7 минут на одной GPU.

Итоговый `step_loss` порядка `1e-3` сам по себе ничего не говорит: в диффузионном
обучении лосс измеряет предсказание шума на случайном таймстепе и слабо связан с
тем, насколько похож получившийся человек. Оценивать приходится генерациями —
чем и занята оставшаяся часть ноутбука.


In [ ]:
!find /kaggle/working -name "*.safetensors"

/kaggle/working/lora_output/checkpoint-250/pytorch_lora_weights.safetensors
/kaggle/working/lora_output/checkpoint-500/pytorch_lora_weights.safetensors
/kaggle/working/lora_output/checkpoint-750/pytorch_lora_weights.safetensors
/kaggle/working/lora_output/checkpoint-1000/pytorch_lora_weights.safetensors
/kaggle/working/lora_output/checkpoint-1500/pytorch_lora_weights.safetensors
/kaggle/working/lora_output/checkpoint-1750/pytorch_lora_weights.safetensors
/kaggle/working/lora_output/checkpoint-2000/pytorch_lora_weights.safetensors

## Пайплайн генерации

`enable_model_cpu_offload` выгружает неиспользуемые компоненты в RAM — иначе SDXL
вместе с VAE не помещается при генерации в 1024.

Сид фиксируется в `generate`: без этого любое сравнение конфигураций
превращается в сравнение сидов, а не настроек.


In [ ]:
from diffusers import StableDiffusionXLPipeline

LORA_PATH = "lora/pytorch_lora_weights.safetensors"   # свой путь к весам

pipe = StableDiffusionXLPipeline.from_pretrained(
    BASE_MODEL, torch_dtype=torch.float16, variant="fp16", use_safetensors=True,
)
pipe.enable_model_cpu_offload()
pipe.enable_vae_slicing()
pipe.set_progress_bar_config(disable=True)
pipe.load_lora_weights(LORA_PATH)


def generate(prompt, negative_prompt="", steps=30, guidance=7.0, lora_scale=0.8, seed=SEED):
    generator = torch.Generator(device="cuda").manual_seed(seed)
    return pipe(
        prompt=prompt,
        negative_prompt=negative_prompt,
        num_inference_steps=steps,
        guidance_scale=guidance,
        generator=generator,
        cross_attention_kwargs={"scale": lora_scale},
    ).images[0]

## Метрика сходства

Визуальная оценка субъективна и, в данном случае, ещё и непубликуема. Считаю
косинусную близость эмбеддингов лиц (FaceNet, InceptionResnetV1, веса vggface2)
между сгенерированным изображением и обучающими фотографиями.

Метрика измеряет только лицо и ничего не говорит о качестве картинки в целом —
она не штрафует за артефакты рук, фона и композиции. Но она позволяет сравнивать
конфигурации между собой числом, а не ощущением.

Верхняя планка — сходство обучающих фотографий между собой. Выше неё ждать
нечего: это разброс одного и того же человека на разных снимках.


In [ ]:
!pip install -q facenet-pytorch

from PIL import Image
from facenet_pytorch import MTCNN, InceptionResnetV1

mtcnn = MTCNN(image_size=160, margin=20, device="cuda")
face_encoder = InceptionResnetV1(pretrained="vggface2").eval().to("cuda")


@torch.no_grad()
def face_embedding(image):
    face = mtcnn(image)
    if face is None:
        return None
    emb = face_encoder(face.unsqueeze(0).to("cuda"))
    return torch.nn.functional.normalize(emb, dim=1).cpu().numpy()[0]


reference = []
for filename in sorted(os.listdir(TRAIN_DIR)):
    if filename.lower().endswith((".png", ".jpg", ".jpeg")):
        emb = face_embedding(Image.open(os.path.join(TRAIN_DIR, filename)).convert("RGB"))
        if emb is not None:
            reference.append(emb)

reference = np.stack(reference)
sim = reference @ reference.T
upper = sim[np.triu_indices(len(reference), k=1)]

print(f"Эталонных эмбеддингов: {len(reference)}")
print(f"Сходство обучающих фото между собой: {upper.mean():.3f} ± {upper.std():.3f}")

In [ ]:
def identity_score(image):
    emb = face_embedding(image)
    if emb is None:
        return np.nan          # лицо не найдено -- считается отдельно
    return float((reference @ emb).mean())

## Подбор силы адаптера

`cross_attention_kwargs={"scale": s}` масштабирует вклад LoRA. Слишком мало —
сходство теряется, слишком много — генерация деградирует и перестаёт реагировать
на остальную часть промпта.

`scale = 0` — базовая линия: адаптер отключён, и виден исходный SDXL на том же
промпте и сиде. Всё, что появляется при росте scale, — вклад дообучения.

Каждая конфигурация прогоняется на четырёх сидах: одна картинка на 1024 — это
слишком шумная оценка, чтобы делать по ней выводы.


In [ ]:
PROMPT = f"a photo of {TRIGGER} woman, portrait, natural light"
NEGATIVE = "blurry, deformed, low quality, bad anatomy"
SEEDS = (1, 2, 3, 4)

rows = []
for scale in (0.0, 0.2, 0.4, 0.6, 0.8, 1.0):
    scores = [identity_score(generate(PROMPT, NEGATIVE, lora_scale=scale, seed=s))
              for s in SEEDS]
    rows.append({
        "lora_scale": scale,
        "identity": np.nanmean(scores),
        "std": np.nanstd(scores),
        "лицо не найдено": int(np.isnan(scores).sum()),
    })

scale_df = pd.DataFrame(rows)
print(scale_df.round(3).to_string(index=False))

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.errorbar(scale_df["lora_scale"], scale_df["identity"], yerr=scale_df["std"],
            marker="o", capsize=4, label="генерации")
ax.axhline(upper.mean(), linestyle="--", color="gray",
           label="сходство обучающих фото между собой")
ax.set_xlabel("lora_scale")
ax.set_ylabel("Средняя близость к эталону")
ax.set_title("Сходство лица в зависимости от силы адаптера")
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## Проверка на переобучение: слушается ли модель промпта

Если адаптер выучил не человека, а обучающие кадры целиком, то при смене
контекста (одежда, фон, сезон) картинка всё равно будет воспроизводить обстановку
обучающих фотографий, а сходство лица останется высоким.

Поэтому смотрю на две вещи сразу: держится ли сходство при смене контекста и не
пропадает ли лицо вовсе. Резкое падение на «сложных» промптах означает, что
адаптер привязан к условиям съёмки.


In [ ]:
CONTEXT_PROMPTS = {
    "нейтральный": f"a photo of {TRIGGER} woman",
    "красное платье": f"a photo of {TRIGGER} woman in a red dress",
    "пляж, закат": f"a photo of {TRIGGER} woman on the beach at sunset",
    "студийный свет": f"a photo of {TRIGGER} woman, studio lighting, black clothes",
    "зима, улица": f"a photo of {TRIGGER} woman wearing a winter jacket in the snow",
}

rows = []
for label, prompt in CONTEXT_PROMPTS.items():
    scores = [identity_score(generate(prompt, NEGATIVE, steps=35, lora_scale=0.8, seed=s))
              for s in SEEDS]
    rows.append({
        "контекст": label,
        "identity": np.nanmean(scores),
        "std": np.nanstd(scores),
        "лицо не найдено": int(np.isnan(scores).sum()),
    })

context_df = pd.DataFrame(rows)
print(context_df.round(3).to_string(index=False))

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.5))
ax.bar(context_df["контекст"], context_df["identity"], yerr=context_df["std"], capsize=4)
ax.axhline(upper.mean(), linestyle="--", color="gray", label="разброс обучающих фото")
ax.axhline(context_df.loc[0, "identity"], linestyle=":", color="tab:red",
           label="нейтральный промпт")
ax.set_ylabel("Близость к эталону")
ax.set_title("Сходство при смене контекста, lora_scale = 0.8")
ax.legend()
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()

## Выводы

Что удалось: SDXL воспроизводит конкретного человека по триггер-слову, при этом
обучается меньше процента параметров, и всё обучение помещается в одну GPU с
16 ГБ за счёт gradient checkpointing, 8-битного Adam и fp16.

Что важно про методику: результат оценивается числом на фиксированных сидах, а
не просмотром удачных картинок. Именно поэтому в ноутбуке есть базовая линия
(`scale = 0`), несколько сидов на конфигурацию и верхняя планка в виде разброса
самих обучающих фотографий.

Ограничения:

- **12 фотографий — мало.** Набор снят в узком диапазоне ракурсов и освещения,
  и модель наследует это ограничение.
- **2000 шагов при 12 изображениях — с запасом.** Чекпоинты сохранялись каждые
  250 шагов именно для того, чтобы выбрать момент остановки сравнением, но до
  сравнения чекпоинтов дело не дошло: сессия Kaggle завершилась, и промежуточные
  веса из `/kaggle/working` не сохранились. Это ошибка планирования, а не вывод
  об эксперименте — при повторе чекпоинты нужно сразу выгружать наружу.
- **Текстовые энкодеры не обучались.** Их обучение обычно улучшает привязку
  триггера к понятию, но повышает риск испортить остальные промпты — на таком
  объёме данных не оправдано.
- **Метрика измеряет только лицо** и не штрафует за артефакты композиции.

Что попробовать дальше: расширить набор до 20-30 кадров с разными ракурсами;
сравнить с DreamBooth и prior preservation loss, который специально борется с
тем, что триггер «съедает» общее понятие; попробовать меньший rank адаптера —
при таком объёме данных большой rank избыточен.
